<a href="https://colab.research.google.com/github/ardatanriverdi/MLProjects/blob/main/spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from matplotlib import pyplot as plt
from sklearn.tree import export_graphviz
import graphviz

In [12]:
df= pd.read_csv('spam.csv')

UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 606-607: invalid continuation byte

# Task
Explain and fix the error in the code to train a text classification model.

## Data preprocessing

### Subtask:
Clean and prepare the text data for model training. This might involve steps like removing punctuation, converting text to lowercase, and tokenization.


**Reasoning**:
The previous attempt to load the CSV failed due to a UnicodeDecodeError. I will try loading the data again, specifying a different encoding that might resolve the issue. Based on common encodings for such errors, I will try 'latin-1'.



In [13]:
try:
    df = pd.read_csv('spam.csv', encoding='latin-1')
except UnicodeDecodeError:
    try:
        df = pd.read_csv('spam.csv', encoding='ISO-8859-1')
    except UnicodeDecodeError:
        print("Could not decode the file with latin-1 or ISO-8859-1. Please check the file encoding.")

if 'df' in locals():
    display(df.head())
    display(df.info())

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


None

**Reasoning**:
The data has been loaded successfully using 'latin-1' encoding. The next steps are to examine the data, handle missing values, convert text to lowercase, remove punctuation, and perform tokenization as per the subtask instructions. I will combine these cleaning and preprocessing steps into a single code block.



In [14]:
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

df.dropna(subset=['message'], inplace=True)

df['message'] = df['message'].str.lower()

df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)

from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
df['message'] = df['message'].apply(word_tokenize)

display(df.head())

<>:8: SyntaxWarning: invalid escape sequence '\w'
<>:8: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-1937972643.py:8: SyntaxWarning: invalid escape sequence '\w'
  df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)
/tmp/ipython-input-1937972643.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(subset=['message'], inplace=True)
/tmp/ipython-input-1937972643.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['message'] = df['message'].str.lower()
/tmp/ipython-input-1937972643.py:8: SettingWithCopyWarning: 
A value

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


**Reasoning**:
The previous command failed because the 'punkt_tab' resource was not found, even though 'punkt' was downloaded. I need to explicitly download 'punkt_tab' or ensure that the 'punkt' download includes the necessary resources for word tokenization. I will try downloading 'punkt' again, as the error message suggests 'punkt_tab' is part of the 'punkt' corpus. I will also address the `SettingWithCopyWarning` by explicitly creating a copy of the DataFrame slice before applying transformations.



In [15]:
import nltk
nltk.download('punkt')

df = df[['v1', 'v2']].copy()
df.columns = ['label', 'message']

df.dropna(subset=['message'], inplace=True)

df['message'] = df['message'].str.lower()

df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)

from nltk.tokenize import word_tokenize
df['message'] = df['message'].apply(word_tokenize)

display(df.head())

<>:11: SyntaxWarning: invalid escape sequence '\w'
<>:11: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-2442790969.py:11: SyntaxWarning: invalid escape sequence '\w'
  df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


KeyError: "None of [Index(['v1', 'v2'], dtype='object')] are in the [columns]"

**Reasoning**:
The previous command failed with a KeyError because the column names were already changed in the first successful loading step. The DataFrame now has 'label' and 'message' columns instead of 'v1' and 'v2'. I need to adjust the column selection to use the correct column names.



In [16]:
df = df[['label', 'message']].copy()

df.dropna(subset=['message'], inplace=True)

df['message'] = df['message'].str.lower()

df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)

from nltk.tokenize import word_tokenize
df['message'] = df['message'].apply(word_tokenize)

display(df.head())

<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:7: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-915773316.py:7: SyntaxWarning: invalid escape sequence '\w'
  df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


**Reasoning**:
The code is still failing because the 'punkt_tab' resource is not found. The error message specifically suggests downloading 'punkt_tab'. I will explicitly download 'punkt_tab' using `nltk.download()`. The rest of the cleaning steps are correct and can be kept.



In [17]:
import nltk
nltk.download('punkt_tab')

df = df[['label', 'message']].copy()

df.dropna(subset=['message'], inplace=True)

df['message'] = df['message'].str.lower()

df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)

from nltk.tokenize import word_tokenize
df['message'] = df['message'].apply(word_tokenize)

display(df.head())

<>:10: SyntaxWarning: invalid escape sequence '\w'
<>:10: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-1862391244.py:10: SyntaxWarning: invalid escape sequence '\w'
  df['message'] = df['message'].str.replace('[^\w\s]', '', regex=True)
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,label,message
0,ham,"[go, until, jurong, point, crazy, available, o..."
1,ham,"[ok, lar, joking, wif, u, oni]"
2,spam,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,ham,"[u, dun, say, so, early, hor, u, c, already, t..."
4,ham,"[nah, i, dont, think, he, goes, to, usf, he, l..."


## Feature extraction

### Subtask:
Convert the text data into numerical features that can be used by machine learning models. Techniques like TF-IDF or Count Vectorization can be used.


**Reasoning**:
Import TfidfVectorizer and join the tokens back into strings in the 'message' column to prepare for vectorization, then instantiate, fit and transform the 'message' column using the TfidfVectorizer.



In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['message_str'] = df['message'].apply(lambda tokens: ' '.join(tokens))

tfidf_vectorizer = TfidfVectorizer()

tfidf_features = tfidf_vectorizer.fit_transform(df['message_str'])

print("Shape of TF-IDF features:", tfidf_features.shape)

Shape of TF-IDF features: (5572, 9522)


## Model selection and training

### Subtask:
Choose a suitable classification model (e.g., Logistic Regression, Decision Tree, Naive Bayes) and train it on the prepared data.


**Reasoning**:
Split the data into training and testing sets, instantiate a Logistic Regression model, and train the model.



In [19]:
X_train, X_test, y_train, y_test = train_test_split(tfidf_features, df['label'], test_size=0.2, random_state=42)

model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

## Model evaluation

### Subtask:
Evaluate the performance of the trained model using appropriate metrics like accuracy, precision, recall, and F1-score.


**Reasoning**:
Use the trained model to predict on the test set and then evaluate the predictions using accuracy, confusion matrix, and classification report.



In [20]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nConfusion Matrix:\n", conf_matrix)
print("\nClassification Report:\n", class_report)

Accuracy: 0.9560538116591928

Confusion Matrix:
 [[965   0]
 [ 49 101]]

Classification Report:
               precision    recall  f1-score   support

         ham       0.95      1.00      0.98       965
        spam       1.00      0.67      0.80       150

    accuracy                           0.96      1115
   macro avg       0.98      0.84      0.89      1115
weighted avg       0.96      0.96      0.95      1115



## Summary:

### Data Analysis Key Findings

*   The initial data loading faced `UnicodeDecodeError`, which was resolved by specifying `latin-1` encoding.
*   The dataset contained irrelevant columns (`Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4`) that were removed.
*   A `SettingWithCopyWarning` occurred during data cleaning, which was fixed by explicitly creating a copy of the DataFrame slice.
*   The `nltk.download('punkt_tab')` command was necessary to resolve a `LookupError` during tokenization.
*   The TF-IDF vectorization resulted in a feature matrix of shape (5572, 9522).
*   The trained Logistic Regression model achieved an accuracy of approximately 95.6% on the test set.
*   The model showed perfect precision (1.00) for 'spam' but lower recall (0.67), indicating it correctly identifies spam when predicted but misses some actual spam messages.

### Insights or Next Steps

*   Investigate techniques to improve the recall for 'spam' messages, potentially by addressing the class imbalance or exploring different models.
*   Consider exploring other feature extraction methods or hyperparameters for the current model to potentially boost performance.
